<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Extracts raw Oracle tables into parquet files under the project's raw data directory.

**Notebook Shape:** 15 cells (14 code, 1 markdown).

**Inputs / Data Sources:**
- `df_v_crg_student_course = pd.read_sql(query, connection)`
- `df_v_acs_grade = pd.read_sql(query, connection)`
- `df_v_add_student_degree_status = pd.read_sql(query, connection)`
- `df_v_acd_degree_course = pd.read_sql(query, connection)`
- `df_v_crg_student_passed_credit = pd.read_sql(query, connection)`

**Outputs / Side Effects:**
- `df_v_crg_student_course.to_parquet(RAW_DIR / "v_crg_student_course_raw.parquet",index=False)`
- `df_v_acs_grade.to_parquet(RAW_DIR / "v_acs_grade.parquet",index=False)`
- `df_v_add_student_degree_status.to_parquet(RAW_DIR / "v_add_student_degree_status.parquet",index=False)`
- `df_v_acd_degree_course.to_parquet(RAW_DIR / "v_acd_degree_course.parquet",index=False)`
- `df_v_crg_student_passed_credit.to_parquet(RAW_DIR / "v_crg_student_passed_credit.parquet",index=False)`

**Logic Flow:**
1. Open an Oracle connection.
2. Run SQL queries for each required source view.
3. Load results into pandas DataFrames.
4. Write raw parquet snapshots.

**Maintainability Notes:** This notebook depends on database credentials and live source data; extraction dates, SQL, and schema versions should be tracked for reproducibility.


## setup and connect with data base

In [1]:
import oracledb
import pandas as pd

from src.paths import RAW_DIR, assert_data_root
from src.io_utils import save_parquet
from src.db_connect import get_connection

# Data-root guard (governance contract 12): refuse to run against a freshly
# created empty tree before extracting any raw table.
assert_data_root()

connection = get_connection()

In [ ]:
cursor = connection.cursor()
cursor.arraysize = 10_000
pd.set_option('display.max_columns', None)

In [ ]:
print("RAW_DIR:", RAW_DIR)


In [5]:
VIEW_NAME = "RAS_USER.V_ADD_ACADEMIC_INFO "

needed_columns = [
    "STUDENT_ID",
    "DIPLOMA_GPA",
    "DIPLOMA_TYPE_ID",
    "DIPLOMA_STATE_ID",
    "DIPLOMA_COUNTRY_SL",
    "DIPLOMA_TYPE_SL"
]
columns_str = ", ".join(needed_columns)
query = f"""
SELECT
    {columns_str}
    ACTIVE
FROM {VIEW_NAME}
WHERE ACTIVE = 'A'
"""

df_v_add_academic_info = pd.read_sql(query, connection)
df_v_add_academic_info.columns = df_v_add_academic_info.columns.str.lower()

save_parquet(df_v_add_academic_info, RAW_DIR / "v_add_academic_info.parquet")

print("Rows loaded:", len(df_v_add_academic_info))
print("Columns:", df_v_add_academic_info.columns.tolist())
print("Saved:", RAW_DIR / "v_add_academic_info.parquet")
display(df_v_add_academic_info.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_27504\2622811198.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v_add_adcademic_info = pd.read_sql(query, connection)


Rows loaded: 32548
Columns: ['student_id', 'diploma_gpa', 'diploma_type_id', 'diploma_state_id', 'diploma_country_sl', 'active']
Saved: D:\AI\Real projects\Academic_Advisor\data\raw\v_crg_student_course_raw.parquet


,student_id,diploma_gpa,diploma_type_id,diploma_state_id,diploma_country_sl,active
0,1.111,60.00,13.111,13.0,سورية,شهادة ثانوية تجارية
1,2.111,56.82,16.111,15.0,سورية,شهادة ثانوية أدبي
2,3.111,60.45,16.111,4.0,سورية,شهادة ثانوية أدبي
3,4.111,50.83,15.111,4.0,سورية,شهادة ثانوية علمي
4,5.111,67.27,13.111,5.0,سورية,شهادة ثانوية تجارية


In [ ]:
VIEW_NAME = "RAS_USER.V_CRG_STUDENT_COURSE"

needed_columns = [
    "STUDENT_COURSE_ID",
    "STUDENT_ID",
    "COURSE_ID",
    "PART_ID",
    "GRADE_ID",
    "FINAL_MARK",
    "POINTS",
    "FINISH_STATUS",
    "COURSE_NAME_SL",
    "REGISTER_STATUS",
    "STUDY_MODE",
    "DEGREE_ID",
    "DEGREE_NAME_SL",
    "FACULTY_ID",
    "COURSE_CREDITS",
    "ACTIVE",
]

query = f"""
SELECT
    STUDENT_COURSE_ID,
    STUDENT_ID,
    COURSE_ID,
    PART_ID,
    GRADE_ID,
    FINAL_MARK,
    POINTS,
    FINISH_STATUS,
    COURSE_NAME_SL,
    REGISTER_STATUS,
    IN_CREDITS,
    IN_GPA,
    IN_AGPA,
    STUDY_MODE,
    DEGREE_ID,
    STUDENT_NAME_SL,
    DEGREE_NAME_SL,
    FACULTY_ID,
    COURSE_CREDITS,
    ACTIVE
FROM {VIEW_NAME}
WHERE ACTIVE = 'A'
  AND STUDY_MODE = 'C'
"""
غ
df_v_crg_student_course = pd.read_sql(query, connection)
df_v_crg_student_course.columns = df_v_crg_student_course.columns.str.lower()

save_parquet(df_v_crg_student_course, RAW_DIR / "v_crg_student_course_raw.parquet")

print("Rows loaded:", len(df_v_crg_student_course))
print("Columns:", df_v_crg_student_course.columns.tolist())
print("Saved:", RAW_DIR / "v_crg_student_course_raw.parquet")
display(df_v_crg_student_course.head())

In [ ]:
VIEW_NAME = "RAS_USER.V_ACS_GRADE"

needed_columns = [
    "GRADE_ID",
    "GRADE_VERSION_ID",
    "VERSION_TITLE_SL",
    "VERSION_NUMBER",
    "FROM_PERCENT",
    "TO_PERCENT",
    "POINTS",
    "FINISH_STATUS",
    "GRADE_SHOW",
    "FROM_SEMESTER_ID",
    "TILL_SEMESTER_ID",
    "ACTIVE",
]

query = f"""
SELECT
    GRADE_ID,
    GRADE_VERSION_ID,
    GRADE_NAME_SL,
    VERSION_TITLE_SL,
    VERSION_NUMBER,
    FROM_PERCENT,
    TO_PERCENT,
    POINTS,
    FINISH_STATUS,
    GRADE_SHOW,
    ACTIVE
FROM {VIEW_NAME}
WHERE ACTIVE = 'A'
"""

df_v_acs_grade = pd.read_sql(query, connection)
df_v_acs_grade.columns = df_v_acs_grade.columns.str.lower()

save_parquet(df_v_acs_grade, RAW_DIR / "v_acs_grade.parquet")

print("Rows loaded:", len(df_v_acs_grade))
print("Columns:", df_v_acs_grade.columns.tolist())
print("Saved:", RAW_DIR / "v_acs_grade.parquet")

display(df_v_acs_grade.head())

In [ ]:
VIEW_NAME = "RAS_USER.V_ADD_STUDENT_DEGREE_STATUS"

needed_columns = [
    "STUDENT_ID",
    "PART_ID",
    "DEGREE_ID",
    "STUDY_MODE",
    "GPA_PERCENT",
    "GPA_POINTS",
    "START_AGPA_PERCENT",
    "START_AGPA_POINTS",
    "END_AGPA_PERCENT",
    "END_AGPA_POINTS",
    "SEMESTER_REG_COURSES",
    "SEMESTER_REG_CREDITS",
    "SEMESTER_PASS_COURSES",
    "SEMESTER_PASS_CREDITS",
    "SEMESTER_FAIL_COURSES",
    "SEMESTER_FAIL_CREDITS",
    "TOTAL_PASS_COURSES",
    "TOTAL_PASS_CREDITS",
    "TOTAL_FAIL_COURSES",
    "TOTAL_FAIL_CREDITS",
    "REG_TOTAL_SEMESTERS",
    "FINISH_STATUS",
    "VERSION_TITLE_SL",
    "START_LEVEL_ID",
    "START_LEVEL_NAME_PL",
]

query = f"""
SELECT
    STUDENT_STATUS_ID,
    STUDENT_ID,
    PART_ID,
    DEGREE_ID,
    START_PART_ID,
    FINISH_PART_ID,
    GRADE_VERSION_ID,
    PERMANENT_STATUS_ID,
    PERMANENT_STATUS_SL,
    STUDY_MODE,
    PREV_GPA_POINTS,
    PREV_GPA_PERCENT,
    GPA_PERCENT,
    GPA_POINTS,
    START_AGPA_PERCENT,
    START_AGPA_POINTS,
    START_TOTAL_IN_COURSES,
    START_TOTAL_IN_CREDITS,
    END_TOTAL_IN_COURSES,
    END_TOTAL_IN_CREDITS,
    END_AGPA_PERCENT,
    END_AGPA_POINTS,
    SEMESTER_REG_COURSES,
    SEMESTER_REG_CREDITS,
    SEMESTER_PASS_COURSES,  
    SEMESTER_PASS_CREDITS,
    SEMESTER_FAIL_COURSES,
    SEMESTER_FAIL_CREDITS,
    SEMESTER_IN_COURSES,
    SEMESTER_IN_CREDITS,
    TOTAL_SEMESTERS,
    TOTAL_REG_COURSES,
    TOTAL_REG_CREDITS,
    TOTAL_PASS_COURSES,
    TOTAL_PASS_CREDITS,
    TOTAL_FAIL_COURSES,
    TOTAL_FAIL_CREDITS,
    REG_TOTAL_SEMESTERS,
    FINISH_STATUS,
    VERSION_TITLE_SL,
    DEGREE_NAME_SL,
    DEGREE_CREDITS_COUNT,
    START_LEVEL_ID,
    START_LEVEL_NAME_PL
FROM {VIEW_NAME}
WHERE STUDY_MODE = 'C'
AND ACTIVE = 'A'
"""

df_v_add_student_degree_status = pd.read_sql(query, connection)
df_v_add_student_degree_status.columns = df_v_add_student_degree_status.columns.str.lower()

save_parquet(df_v_add_student_degree_status, RAW_DIR / "v_add_student_degree_status.parquet")

print("Rows loaded:", len(df_v_add_student_degree_status))
print("Columns:", df_v_add_student_degree_status.columns.tolist())
print("Saved:", RAW_DIR / "v_add_student_degree_status.parquet")

display(df_v_add_student_degree_status.head())

In [ ]:
VIEW_NAME = "RAS_USER.V_ACD_DEGREE_COURSE"

needed_columns = [
    "DEGREE_COURSE_ID",
    "COURSE_ID",
    "COURSE_NAME_SL",
    "DEGREE_ID",
    "DEGREE_NAME_SL",
    "YEAR_ORDER",
    "SEMESTER_ORDER",
    "COURSE_CREDITS",
    "ACTIVE",
    "CREDITS_COUNT",
    "FACULTY_ID",
    "REQUIRED_CREDITS",
    "REQUIREMENT_TYPE_ID",
    "REQUIREMENT_TYPE_SL",
    "REQ_DEGREE_ID",
]

query = f"""
SELECT
    DEGREE_COURSE_ID,
    COURSE_ID,
    DEGREE_ID,
    COURSE_TYPE_ID,
    REQUIREMENT_TYPE_ID,
    REQUIREMENT_TYPE_SL,
    COURSE_NAME_SL,
    COURSE_OFFICIAL_SL,
    DEGREE_NAME_SL,
    YEAR_ORDER,
    SEMESTER_ORDER,
    COURSE_CREDITS,
    ACTIVE,
    CREDITS_COUNT,
FROM {VIEW_NAME}
WHERE ACTIVE = 'A'
"""

df_v_acd_degree_course = pd.read_sql(query, connection)
df_v_acd_degree_course.columns = df_v_acd_degree_course.columns.str.lower()

save_parquet(df_v_acd_degree_course, RAW_DIR / "v_acd_degree_course.parquet")

print("Rows loaded:", len(df_v_acd_degree_course))
print("Columns:", df_v_acd_degree_course.columns.tolist())
print("Saved:", RAW_DIR / "v_acd_degree_course.parquet")

display(df_v_acd_degree_course.head())

In [ ]:
VIEW_NAME = "RAS_USER.V_CRG_STUDENT_PASSED_CREDIT"

query = f"""
SELECT
    STUDENT_ID,
    REQUIREMENT_TYPE_PL,
    REQUIREMENT_TYPE_SL,
    CREDITS_COUNT,
    PASSED_CREDIT
FROM {VIEW_NAME}
"""

df_v_crg_student_passed_credit = pd.read_sql(query, connection)
df_v_crg_student_passed_credit.columns = df_v_crg_student_passed_credit.columns.str.lower()

save_parquet(df_v_crg_student_passed_credit, RAW_DIR / "v_crg_student_passed_credit.parquet")

print("Rows loaded:", len(df_v_crg_student_passed_credit))
print("Columns:", df_v_crg_student_passed_credit.columns.tolist())
print("Saved:", RAW_DIR / "v_crg_student_passed_credit.parquet")

display(df_v_crg_student_passed_credit.head())

In [ ]:
VIEW_NAME = "RAS_USER.V_SCH_COURSE_OFFER"

query = f"""
SELECT
    LEVEL_CATEGORY_ID,
    PART_ID,
    DEPARTMENT_ID,
    FACULTY_ID,
    COURSE_ID,
    COURSE_TYPE_ID,
    COURSE_NAME_SL,
    FACULTY_NAME_SL,
    DEPARTMENT_NAME_SL,
    COURSE_CREDITS,
    COURSE_STATUS,
    ALLOW_REGISTER
FROM {VIEW_NAME}
WHERE ACTIVE = 'A'
"""

df_v_sch_course_offers = pd.read_sql(query, connection)
df_v_sch_course_offers.columns = df_v_sch_course_offers.columns.str.lower()

save_parquet(df_v_sch_course_offers, RAW_DIR / "v_sch_course_offers.parquet")

print("Rows loaded:", len(df_v_sch_course_offers))
print("Columns:", df_v_sch_course_offers.columns.tolist())
print("Saved:", RAW_DIR / "v_sch_course_offers.parquet")

display(df_v_sch_course_offers.head())

In [ ]:
VIEW_NAME = "RAS_USER.V_CRG_STD_COR_TEMP_REQUEST"


query = f"""
SELECT
ACTIVE,
REQUIREMENT_TYPE_ID,
IS_REQUESTABLE,
PASSED_PREREQUISITES,
STATUS_REASON_SL,
STATUS_REASON_CODE,
FINAL_MARK,
SEMESTER_ORDER,
YEAR_ORDER,
COURSE_CREDITS,
COURSE_NAME_SL, 
COURSE_ID,
CREDITS_COUNT,
CREDITS_TILL_GRAD,
STUDENT_ID,
STD_COR_TEMP_REQUEST_ID
FROM {VIEW_NAME}
"""

df_vcrg_std_cor_temp_request = pd.read_sql(query, connection)
df_vcrg_std_cor_temp_request.columns = df_vcrg_std_cor_temp_request.columns.str.lower()

save_parquet(df_vcrg_std_cor_temp_request, RAW_DIR / "v_crg_std_cor_temp_request.parquet")

print("Rows loaded:", len(df_vcrg_std_cor_temp_request))
print("Columns:", df_vcrg_std_cor_temp_request.columns.tolist())
print("Saved:", RAW_DIR / "v_crg_std_cor_temp_request.parquet")

display(df_vcrg_std_cor_temp_request.head())

In [ ]:
VIEW_NAME = "RAS_USER.V_CRG_STD_COR_TEMP_REQUEST"

needed_columns = [
"ACTIVE",
"REQUIREMENT_TYPE_ID",
"IS_REQUESTABLE",
"PASSED_PREREQUISITES",
"LAST_REGISTER_SEMESTER"
"STATUS_REASON_SL",
"STATUS_REASON_CODE",
"FINAL_MARK",
"SEMESTER_ORDER",
"YEAR_ORDER",
"COURSE_CREITS",
"COURSE_NAME_SL", 
"COURSE_ID",
"CREDITS_COUNT",
"CREDITS_TILL_GRAD",
"STUDENT_ID",
"STD_COR_TEMP_REQUEST_ID"
]

query = f"""
SELECT
    STD_COR_TEMP_REQUEST_ID,
    STUDENT_ID,
    PART_ID,
    COURSE_ID,
    COURSE_TYPE_ID,
    REQUIREMENT_TYPE_ID,

    ALLOW_REGISTER,
    IS_REQUESTABLE,
    GPA_PERCENT,
    GPA_POINTS,
    END_AGPA_POINTS,
    CREDITS_TILL_GRAD,
    REQUIREMENT_PASSED_CREDITS,
    CREDITS_COUNT,
    COURSE_CREDITS,
    YEAR_ORDER,
    SEMESTER_ORDER,
    REGISTER_STATUS,
    FINISH_STATUS
FROM RAS_USER.V_CRG_STD_COR_TEMP_REQUEST
ORDER BY
    STUDENT_ID,
    PART_ID,
    COURSE_ID,
    REQUIREMENT_TYPE_ID
"""

df_vcrg_std_cor_temp_request = pd.read_sql(query, connection)
df_vcrg_std_cor_temp_request.columns = df_vcrg_std_cor_temp_request.columns.str.lower()

save_parquet(df_vcrg_std_cor_temp_request, RAW_DIR / "v_crg_std_cor_temp_request.parquet")

print("Rows loaded:", len(df_vcrg_std_cor_temp_request))
print("Columns:", df_vcrg_std_cor_temp_request.columns.tolist())
print("Saved:", RAW_DIR / "v_crg_std_cor_temp_request.parquet")

display(df_vcrg_std_cor_temp_request.head())

In [ ]:
VIEW_NAME= "RAS_USER.V_COR_COURSE_PREREQUISITE "
query = f"""
SELECT*
FROM {VIEW_NAME}
"""
df_v_cor_course_prerequisite = pd.read_sql(query, connection)
df_v_cor_course_prerequisite.columns = df_v_cor_course_prerequisite.columns.str.lower()

save_parquet(df_v_cor_course_prerequisite, RAW_DIR / "v_cor_course_prerequisite.parquet")

print("Rows loaded:", len(df_v_cor_course_prerequisite))
print("Columns:", df_v_cor_course_prerequisite.columns.tolist())
print("Saved:", RAW_DIR / "v_cor_course_prerequisite.parquet")
display(df_v_cor_course_prerequisite.head())

In [ ]:
cursor.close()